In [0]:
import pandas as pd
import numpy as np
import unicodedata
import re
from collections import defaultdict

# Tabla histórica final con features + ranking FIFA
historical_df = spark.table("workspace.gold.training_dataset_features_fifa").toPandas()

# Partidos del Mundial 2026 sin marcador
worldcup_2026 = spark.table("workspace.silver.worldcup_2026_group_matches").toPandas()

# Ranking FIFA limpio desde Silver
fifa = spark.table("workspace.silver.fifa_rankings_clean").toPandas()

# Convertir fechas
historical_df["date"] = pd.to_datetime(historical_df["date"])
worldcup_2026["date"] = pd.to_datetime(worldcup_2026["date"])
fifa["fifa_rank_date"] = pd.to_datetime(fifa["fifa_rank_date"])

print("Históricos con FIFA:", historical_df.shape)
print("Mundial 2026:", worldcup_2026.shape)
print("Ranking FIFA:", fifa.shape)

display(worldcup_2026.head())

In [0]:

#Como ya tenemos el modelo probado, podemos entrenarlo con todos los partidos históricos desde 2000 para predecir 2026.

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

features = [
    "tournament_weight",
    "is_world_cup",
    "is_world_cup_qualifier",
    "is_friendly",
    "is_neutral",
    
    "home_recent_games",
    "away_recent_games",
    
    "home_recent_win_rate",
    "away_recent_win_rate",
    "home_recent_draw_rate",
    "away_recent_draw_rate",
    "home_recent_loss_rate",
    "away_recent_loss_rate",
    
    "home_recent_avg_goals_for",
    "away_recent_avg_goals_for",
    "home_recent_avg_goals_against",
    "away_recent_avg_goals_against",
    "home_recent_avg_goal_diff",
    "away_recent_avg_goal_diff",
    
    "recent_win_rate_diff",
    "recent_draw_rate_diff",
    "recent_loss_rate_diff",
    "recent_avg_goals_for_diff",
    "recent_avg_goals_against_diff",
    "recent_avg_goal_diff_diff",
    "recent_games_diff",

    # FIFA Ranking features
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "fifa_rank_diff",
    "fifa_points_diff"
]

# Usamos partidos desde 2000 para entrenar el modelo final
df_model = historical_df[historical_df["year"] >= 2000].copy()

# Rellenar nulos para evitar errores
df_model[features] = df_model[features].fillna(0)

X = df_model[features]
y = df_model["target"]

final_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        batch_size=32,
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.2,
        n_iter_no_change=15,
        random_state=42
    ))
])

final_model.fit(X, y)

print("Modelo final con ranking FIFA entrenado.")
print("Filas usadas para entrenamiento:", X.shape[0])
print("Variables usadas:", X.shape[1])

In [0]:
historical_sorted = historical_df.sort_values("date").copy()

team_history = defaultdict(list)

for _, row in historical_sorted.iterrows():
    home_team = row["home_team"]
    away_team = row["away_team"]
    
    home_score = row["home_score"]
    away_score = row["away_score"]
    
    if home_score > away_score:
        home_result = "win"
        away_result = "loss"
    elif home_score < away_score:
        home_result = "loss"
        away_result = "win"
    else:
        home_result = "draw"
        away_result = "draw"
    
    team_history[home_team].append({
        "result": home_result,
        "goals_for": home_score,
        "goals_against": away_score
    })
    
    team_history[away_team].append({
        "result": away_result,
        "goals_for": away_score,
        "goals_against": home_score
    })


def calculate_recent_stats(history, n=5):
    recent = history[-n:]
    
    if len(recent) == 0:
        return {
            "recent_games": 0,
            "recent_win_rate": 0,
            "recent_draw_rate": 0,
            "recent_loss_rate": 0,
            "recent_avg_goals_for": 0,
            "recent_avg_goals_against": 0,
            "recent_avg_goal_diff": 0
        }
    
    games = len(recent)
    wins = sum(1 for match in recent if match["result"] == "win")
    draws = sum(1 for match in recent if match["result"] == "draw")
    losses = sum(1 for match in recent if match["result"] == "loss")
    
    goals_for = sum(match["goals_for"] for match in recent)
    goals_against = sum(match["goals_against"] for match in recent)
    
    return {
        "recent_games": games,
        "recent_win_rate": wins / games,
        "recent_draw_rate": draws / games,
        "recent_loss_rate": losses / games,
        "recent_avg_goals_for": goals_for / games,
        "recent_avg_goals_against": goals_against / games,
        "recent_avg_goal_diff": (goals_for - goals_against) / games
    }

print("Historial de selecciones creado correctamente.")
print("Número de selecciones en historial:", len(team_history))

In [0]:
def name_key(name):
    if pd.isna(name):
        return None
    
    name = str(name).strip().lower()
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = name.replace("&", " and ")
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    
    mapping = {
        "ivory coast": "cote divoire",
        "cote d ivoire": "cote divoire",
        "cote divoire": "cote divoire",
        "dr congo": "congo dr",
        "democratic republic of congo": "congo dr",
        "south korea": "korea republic",
        "north korea": "korea dpr",
        "czech republic": "czechia",
        "usa": "united states",
        "united states of america": "united states",
        "turkey": "turkiye",
        "curacao": "curacao",
        "cape verde": "cabo verde",
        "swaziland": "eswatini",
        "macedonia": "north macedonia",
        "iran": "ir iran",
        "kyrgyzstan": "kyrgyz republic",
        "brunei": "brunei darussalam",
        "east timor": "timor leste",
        "bosnia and herzegovina": "bosnia and herzegovina"
    }
    
    return mapping.get(name, name)


fifa = fifa.copy()
fifa["country_key"] = fifa["country_full"].apply(name_key)

def get_latest_fifa(team_name, match_date):
    team_key = name_key(team_name)
    
    team_fifa = fifa[
        (fifa["country_key"] == team_key) &
        (fifa["fifa_rank_date"] <= match_date)
    ].sort_values("fifa_rank_date")
    
    if team_fifa.empty:
        return {
            "fifa_rank": np.nan,
            "fifa_points": np.nan,
            "fifa_rank_date": pd.NaT
        }
    
    latest = team_fifa.iloc[-1]
    
    return {
        "fifa_rank": latest["fifa_rank"],
        "fifa_points": latest["fifa_points"],
        "fifa_rank_date": latest["fifa_rank_date"]
    }

print("Funciones FIFA listas.")

In [0]:
prediction_rows = []

for _, row in worldcup_2026.iterrows():
    home_team = row["home_team"]
    away_team = row["away_team"]
    match_date = row["date"]
    
    home_stats = calculate_recent_stats(team_history[home_team], n=5)
    away_stats = calculate_recent_stats(team_history[away_team], n=5)
    
    home_fifa = get_latest_fifa(home_team, match_date)
    away_fifa = get_latest_fifa(away_team, match_date)
    
    feature_row = row.to_dict()
    
    feature_row["year"] = match_date.year
    
    feature_row["tournament_weight"] = 5
    feature_row["is_world_cup"] = 1
    feature_row["is_world_cup_qualifier"] = 0
    feature_row["is_friendly"] = 0
    feature_row["is_neutral"] = int(row["neutral"])
    
    for key, value in home_stats.items():
        feature_row[f"home_{key}"] = value
    
    for key, value in away_stats.items():
        feature_row[f"away_{key}"] = value
    
    feature_row["recent_win_rate_diff"] = home_stats["recent_win_rate"] - away_stats["recent_win_rate"]
    feature_row["recent_draw_rate_diff"] = home_stats["recent_draw_rate"] - away_stats["recent_draw_rate"]
    feature_row["recent_loss_rate_diff"] = home_stats["recent_loss_rate"] - away_stats["recent_loss_rate"]
    feature_row["recent_avg_goals_for_diff"] = home_stats["recent_avg_goals_for"] - away_stats["recent_avg_goals_for"]
    feature_row["recent_avg_goals_against_diff"] = home_stats["recent_avg_goals_against"] - away_stats["recent_avg_goals_against"]
    feature_row["recent_avg_goal_diff_diff"] = home_stats["recent_avg_goal_diff"] - away_stats["recent_avg_goal_diff"]
    feature_row["recent_games_diff"] = home_stats["recent_games"] - away_stats["recent_games"]
    
    feature_row["fifa_rank_home"] = home_fifa["fifa_rank"]
    feature_row["fifa_rank_away"] = away_fifa["fifa_rank"]
    feature_row["fifa_points_home"] = home_fifa["fifa_points"]
    feature_row["fifa_points_away"] = away_fifa["fifa_points"]
    feature_row["fifa_rank_date_home"] = home_fifa["fifa_rank_date"]
    feature_row["fifa_rank_date_away"] = away_fifa["fifa_rank_date"]
    
    feature_row["fifa_rank_diff"] = feature_row["fifa_rank_away"] - feature_row["fifa_rank_home"]
    feature_row["fifa_points_diff"] = feature_row["fifa_points_home"] - feature_row["fifa_points_away"]
    
    prediction_rows.append(feature_row)

worldcup_features = pd.DataFrame(prediction_rows)

worldcup_features[features] = worldcup_features[features].fillna(0)

display(worldcup_features[[
    "date",
    "home_team",
    "away_team",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "fifa_rank_diff",
    "fifa_points_diff",
    "fifa_rank_date_home",
    "fifa_rank_date_away"
]].head(30))

In [0]:
# Validar que las columnas del modelo existan en worldcup_features
missing_features = [col for col in features if col not in worldcup_features.columns]

if missing_features:
    raise ValueError(f"Faltan estas columnas en worldcup_features: {missing_features}")

# Rellenar nulos antes de predecir
worldcup_features[features] = worldcup_features[features].fillna(0)

# Predicciones
worldcup_proba = final_model.predict_proba(worldcup_features[features])
worldcup_pred = final_model.predict(worldcup_features[features])

worldcup_predictions = worldcup_features[[
    "date",
    "home_team",
    "away_team",
    "tournament",
    "city",
    "country",
    "neutral",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "fifa_rank_diff",
    "fifa_points_diff"
]].copy()

worldcup_predictions["predicted_class"] = worldcup_pred
worldcup_predictions["prob_home_win"] = worldcup_proba[:, 0]
worldcup_predictions["prob_draw"] = worldcup_proba[:, 1]
worldcup_predictions["prob_away_win"] = worldcup_proba[:, 2]

class_mapping_reverse = {
    0: "home_win",
    1: "draw",
    2: "away_win"
}

worldcup_predictions["predicted_result"] = worldcup_predictions["predicted_class"].map(class_mapping_reverse)

display(worldcup_predictions)

In [0]:
worldcup_predictions_spark = spark.createDataFrame(worldcup_predictions)

worldcup_predictions_spark.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.gold.worldcup_2026_match_predictions")

print("Predicciones de partidos del Mundial 2026 guardadas en Gold.")

In [0]:
worldcup_predictions["prob_sum"] = (
    worldcup_predictions["prob_home_win"] +
    worldcup_predictions["prob_draw"] +
    worldcup_predictions["prob_away_win"]
)

display(worldcup_predictions[[
    "home_team",
    "away_team",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "prob_home_win",
    "prob_draw",
    "prob_away_win",
    "prob_sum",
    "predicted_result"
]].head(30))

In [0]:
worldcup_predictions["max_probability"] = worldcup_predictions[[
    "prob_home_win",
    "prob_draw",
    "prob_away_win"
]].max(axis=1)

most_confident = worldcup_predictions.sort_values("max_probability", ascending=False)

display(most_confident[[
    "home_team",
    "away_team",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "predicted_result",
    "prob_home_win",
    "prob_draw",
    "prob_away_win",
    "max_probability"
]].head(20))

In [0]:
display(worldcup_predictions[[
    "home_team",
    "away_team",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "prob_home_win",
    "prob_draw",
    "prob_away_win",
    "predicted_result"
]].sort_values("prob_home_win", ascending=False).head(20))

In [0]:
display(worldcup_predictions[[
    "home_team",
    "away_team",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "prob_home_win",
    "prob_draw",
    "prob_away_win",
    "predicted_result"
]].sort_values("prob_away_win", ascending=False).head(20))